##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `1616714`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 0 Setup

In [29]:
import pandas as pd
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support
)



categorical_features = [ "workclass", "education", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]
numerical_features = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
ALL_FEATURES = categorical_features + numerical_features
TARGET = "income"

EPS = 1e-9
ALPHA = 1.0

train_df = pd.read_csv('Assignment1_data/adult_supervised_train.csv')
unlabelled_df = pd.read_csv("Assignment1_data/adult_unlabelled.csv")
test_df = pd.read_csv("Assignment1_data/adult_test.csv")

def clean_data(df, concept = True):

    df.replace('?', np.nan, inplace=True)
    if df["fnlwgt"].dtype == "object": 
        df.drop("fnlwgt", inplace = True, axis = 1)
    df.dropna(subset = ALL_FEATURES, inplace = True)
    # encode concept to binary high income or not
    if concept:
        df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

    return df

## 0.5. Helpers

In [30]:
# mean and variances for each class of the continuous features
def gaussian_params(df):
    params = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        params[column] = {
            'mean': subset[numerical_features].mean(),
            'var': subset[numerical_features].var()
        }
    return params

# mean and variance for each class of the categorical features
def categorical_params(df):
    probs = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        probs[column] = {}
        for feature in categorical_features:
            value_counts = subset[feature].value_counts()
            total_count = len(subset)
            probs[column][feature] = (value_counts / total_count).to_dict()
    return probs
    
def gaussian_log_probs(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)



class MixedNaiveBayes:

    def __init__(self):
        # initialize both GaussianNB and CategoricalNB
        self.gnb = GaussianNB()
        self.cnb = CategoricalNB(alpha = ALPHA)

    def fit(self, X_cat, X_cont, y):
        # fit both models separately
        self.gnb.fit(X_cont, y)
        self.cnb.fit(X_cat, y)
        self.classes = self.gnb.classes_

    def predict_log_proba(self, X_cont, X_cat):
        # return combined log probabilities
        log_prob_cont = self.gnb.predict_log_proba(X_cont)
        log_prob_cat = self.cnb.predict_log_proba(X_cat)
        log_class_prior = np.log(self.gnb.class_prior_)
        return log_prob_cat + log_prob_cont - log_class_prior

    def predict(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        return self.classes[np.argmax(log_probs, axis=1)]

    def posterior_ratio(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        
        return np.exp(log_probs[:, 1] - log_probs[:, 0])




## 1. Supervised model training


In [31]:
cleaned_train_df = clean_data(train_df)
cleaned_unlabelled_df = clean_data(unlabelled_df, concept=False)
cleaned_test_df = clean_data(test_df, concept=False)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

x_cat_train = encoder.fit_transform(cleaned_train_df[categorical_features]).astype(int)
x_cat_train = x_cat_train + 1

x_cont_train = cleaned_train_df[numerical_features].values
y_train = cleaned_train_df[TARGET].values

min_categories = [len(cats) + 1 for cats in encoder.categories_]

model = MixedNaiveBayes()
model.cnb = CategoricalNB(alpha=ALPHA, min_categories=min_categories)
model.fit(x_cat_train, x_cont_train, y_train)

# calculate priors
class_priors = dict(zip(model.classes, model.gnb.class_prior_))

# Print each class and its prior
print("Q1.1 Class Priors\n")
for class_label, prior_probability in class_priors.items():
    print(f"Class {class_label}: {prior_probability:.4f}")

print("\nQ1.2 Continuous Feature analysis")

means = model.gnb.theta_
vars_ = model.gnb.var_

feature_scores = []

for i, col in enumerate(numerical_features):
    mu0, mu1 = means[0][i], means[1][i]
    var0, var1 = vars_[0][i], vars_[1][i]

    std1, std2 = np.sqrt(var0), np.sqrt(var1)
    
    # Standardised difference
    separation = abs(mu1 - mu0) / np.sqrt((var0 + var1) / 2)
    

    feature_scores.append((col, mu0, mu1, std1, std2, separation))

# Sort by separation
feature_scores.sort(key=lambda x: x[5], reverse=True)

print("\nFeature | Mean(≤50K) | Mean(>50K) | Std Dev(≤50K) | Std Dev(>50K) | Separation")
for f in feature_scores:
    print(f"{f[0]:20} {f[1]:10.2f} {f[2]:10.2f} {f[3]:10.4f} {f[4]:10.4f} {f[5]:10.4f}")


print("\nQ1.3 Categorical Feature analysis (R values)")

feature_log_probs = model.cnb.feature_log_prob_
ratios = []

for i, col in enumerate(categorical_features):
    categories = encoder.categories_[i]

    for j, val in enumerate(categories):
        log_p1 = feature_log_probs[i][1][j]
        log_p0 = feature_log_probs[i][0][j]

        ratio = np.exp(log_p1 - log_p0)
        ratios.append((col, val, ratio))

ratios_sorted = sorted(ratios, key=lambda x: x[2], reverse=True)

print("\nTop 5 for >50K:")
for r in ratios_sorted[:5]:
    print(f"{r[0]} | {r[1]} | {r[2]:.4f}")

print("\nTop 5 for ≤50K:")
for r in ratios_sorted[-5:]:
    print(f"{r[0]} | {r[1]} | {r[2]:.4f}")

Q1.1 Class Priors

Class 0: 0.7541
Class 1: 0.2459

Q1.2 Continuous Feature analysis

Feature | Mean(≤50K) | Mean(>50K) | Std Dev(≤50K) | Std Dev(>50K) | Separation
education-num              9.63      11.59     2.4478     2.3650     0.8166
age                       37.05      43.94    13.7119    10.3027     0.5679
hours-per-week            39.43      45.64    11.9108    10.3961     0.5558
capital-gain             157.67    3607.15  1017.8632 13616.6230     0.3573
capital-loss              55.97     202.36   316.0201   603.9910     0.3037

Q1.3 Categorical Feature analysis (R values)

Top 5 for >50K:
education | Some-college | 7.9572
education | HS-grad | 7.1341
marital-status | Married-civ-spouse | 4.2874
education | Preschool | 3.5545
native-country | Thailand | 3.3825

Top 5 for ≤50K:
education | Assoc-acdm | 0.1508
relationship | Own-child | 0.1445
occupation | Priv-house-serv | 0.1354
occupation | Prof-specialty | 0.0838
relationship | Unmarried | 0.0522


## 2. Supervised model evaluation

In [32]:
print("Q2.1 Classifier accuracy")

x_cont_test = cleaned_test_df[numerical_features].values
x_cat_test = encoder.transform(cleaned_test_df[categorical_features]).astype(int)

unseen_category = (x_cat_test == -1)
rows_with_unseen = np.any(unseen_category, axis=1)
print(f"Rows with unseen categories: {np.sum(rows_with_unseen)}")

x_cat_test = x_cat_test + 1
y_predictions = model.predict(x_cont_test, x_cat_test)
y_test = cleaned_test_df[TARGET].values

# Check for remaining unseen categories (should be 0)
unseen_category = np.isnan(x_cat_test)
rows_with_unseen = np.any(unseen_category, axis=1)








Q2.1 Classifier accuracy
Rows with unseen categories: 1


## 3. Extending the model with semi-supervised training

## 4. Supervised model evaluation